In [2]:
import os
from pathlib import Path

print("Current working directory:")
print(os.getcwd())

print("\nFiles in current directory:")
for f in os.listdir():
    print("-", f)

Current working directory:
/home/vramon/notebooks/WebNLG_MULTI/code_translation/Registry_triples

Files in current directory:
- entity_translations_ca.csv
- entity_translations_ca.revoted.csv
- entity_translations_gl_revoted_with_output_token.csv
- entity_translations_gl.csv
- entity_translations_gl.revoted.csv
- entity_translations_ca_revoted_with_output_token.csv
- relation_translations_ca_revoted_with_output_token.csv
- entity_translations_eu_revoted_with_output_token.csv
- entity_translations_eu.revoted.csv
- relation_translations_ca.csv
- revoting_v2.ipynb
- entity_translations_eu.csv
- relation_translations_gl_revoted_with_output_token.csv
- output_triples_format.ipynb
- relation_translations_gl.revoted.csv
- revoting.ipynb
- relation_translations_eu_revoted_with_output_token.csv
- relation_translations_eu.revoted.csv
- relation_translations_ca.revoted.csv
- relation_translations_gl.csv
- relation_translations_eu.csv
- .ipynb_checkpoints


In [3]:
# Re-build triplesets tokens for cooficial languages (ca, gl, eu)
# - Entities: output_token = final_text with spaces -> underscores (e.g., "Leo García" -> "Leo_García")
# - Relations: output_token = final_text in CamelCase (TitleCase words, remove spaces) (e.g., "fecha de nacimiento" -> "FechaDeNacimiento")

import re
from pathlib import Path
import pandas as pd

# ---------
# Config
# ---------
ENTITY_CSV = "/home/vramon/notebooks/WebNLG_BT/Registry_triples/entity_translations_ca_en.revoted.csv"
RELATION_CSV = "/home/vramon/notebooks/WebNLG_BT/Registry_triples/relation_translations_ca_en.revoted.csv"

OUT_ENTITY_CSV = "/home/vramon/notebooks/WebNLG_BT/Registry_triples/entity_translations_ca_en_revoted_with_output_token.csv"
OUT_RELATION_CSV = "/home/vramon/notebooks/WebNLG_BT/Registry_triples/relation_translations_ca_en_revoted_with_output_token.csv"

# ---------
# Helpers
# ---------
_ws_re = re.compile(r"\s+")

def normalize_whitespace(s: str) -> str:
    # collapse multiple whitespace (including tabs/newlines) into single spaces and strip ends
    return _ws_re.sub(" ", s).strip()

def entity_output_token(final_text: str) -> str:
    # spaces -> underscores, preserving accents and punctuation
    s = normalize_whitespace(final_text)
    return s.replace(" ", "_")

def relation_output_token(final_text: str) -> str:
    # Convert to CamelCase by splitting on whitespace and title-casing each token, then joining.
    # Example: "fecha de nacimiento" -> "FechaDeNacimiento"
    s = normalize_whitespace(final_text)
    parts = s.split(" ") if s else []
    # Python's str.title() can behave oddly for some cases; using first-char upper + rest as-is is safer.
    def cap(word: str) -> str:
        return word[:1].upper() + word[1:] if word else word
    return "".join(cap(w) for w in parts)

def require_columns(df: pd.DataFrame, cols: list[str], name: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

# ---------
# Load
# ---------
entity_path = Path(ENTITY_CSV)
relation_path = Path(RELATION_CSV)

entities = pd.read_csv(entity_path)
relations = pd.read_csv(relation_path)

require_columns(entities, ["final_text"], "Entities CSV")
require_columns(relations, ["final_text"], "Relations CSV")

# ---------
# Create output_token
# ---------
# Handle missing final_text safely (keep as NA)
entities["output_token"] = entities["final_text"].astype("string").apply(
    lambda x: pd.NA if pd.isna(x) else entity_output_token(str(x))
)

relations["output_token"] = relations["final_text"].astype("string").apply(
    lambda x: pd.NA if pd.isna(x) else relation_output_token(str(x))
)

# ---------
# Sanity checks
# ---------
print("Entities preview:")
display(entities[["lang", "source_token", "final_text", "output_token"]].head(10))

print("\nRelations preview:")
display(relations[["lang", "source_token", "final_text", "output_token"]].head(10))

print("\nMissing output_token counts:")
print("entities:", entities["output_token"].isna().sum())
print("relations:", relations["output_token"].isna().sum())

# Optional: check duplicates within each lang (useful for triple building)
dup_ent = entities.dropna(subset=["output_token"]).duplicated(subset=["lang", "output_token"]).sum()
dup_rel = relations.dropna(subset=["output_token"]).duplicated(subset=["lang", "output_token"]).sum()
print("\nDuplicate (lang, output_token) counts:")
print("entities:", dup_ent)
print("relations:", dup_rel)

# ---------
# Save
# ---------
entities.to_csv(OUT_ENTITY_CSV, index=False)
relations.to_csv(OUT_RELATION_CSV, index=False)

print("\nWrote:")
print("-", OUT_ENTITY_CSV)
print("-", OUT_RELATION_CSV)

Entities preview:


,lang,source_token,final_text,output_token
0,en,"""100305.0""(minuts)","""100305.0""(minutes)","""100305.0""(minutes)"
1,en,"""13017.0""(minuts)","""13017.0""(minutes)","""13017.0""(minutes)"
2,en,"""52.0""(minuts)","""52.0""(minutes)","""52.0""(minutes)"
3,en,"""8820.0""(minuts)","""8820.0""(minutes)","""8820.0""(minutes)"
4,en,"""Alvinegro","""Alvinegro","""Alvinegro"
5,en,&_min;7,& min;7,&_min;7
6,en,'Til_Death_Do_Us_Part_(EP)_(en_anglès).,'Til Death Do Us Part (EP) (in English).,'Til_Death_Do_Us_Part_(EP)_(in_English).
7,en,(10)_Higiea,(10) Hygiene,(10)_Hygiene
8,en,(1000)_Piazzia,(1000) Piazzia,(1000)_Piazzia
9,en,(1001)_Gaussia,(1001) Gaussia,(1001)_Gaussia



Relations preview:


,lang,source_token,final_text,output_token
0,en,%1QuilòmetresQuadratsamountInUnits(integer),%1 Square Kilometersamount In Units(integer),%1SquareKilometersamountInUnits(integer)
1,en,1rNúmeroDePista,1st Track Number,1stTrackNumber
2,en,3aLongitudDeLaPistaEnPeus,3rd Track Length In Feet,3rdTrackLengthInFeet
3,en,5èNúmeroDePista,5th Track Number,5thTrackNumber
4,en,AL'oficinaMentreElGovernador,In the office While the governor,InTheOfficeWhileTheGovernor
5,en,AL'oficinaMentreElPresident,In the office While the President,InTheOfficeWhileThePresident
6,en,AL'oficinaMentreElVicepresident,In the office While the Vice President,InTheOfficeWhileTheVicePresident
7,en,AL'oficinaMentreMonarch,In the office While Monarch,InTheOfficeWhileMonarch
8,en,Abreviació,Abbreviation,Abbreviation
9,en,Adreça,Address,Address



Missing output_token counts:
entities: 0
relations: 1

Duplicate (lang, output_token) counts:
entities: 2
relations: 3

Wrote:
- /home/vramon/notebooks/WebNLG_BT/Registry_triples/entity_translations_ca_en_revoted_with_output_token.csv
- /home/vramon/notebooks/WebNLG_BT/Registry_triples/relation_translations_ca_en_revoted_with_output_token.csv
